# Reduced Dataset Composition
- In this notebook we present the process that we apply to compose a dataset with a reduced number of rows and columns, starting from the complete dataset (containing all the features and data point obtained after the merging and labelling steps). 
- We show the methodology we apply, not only for investigating the features, but also to prepare the data to be used by ML algorithms.
- The reduction of the rows that completes the composition of the reduced dataset is reported in the <strong>"NAME OF THE NOTEBOOK" </strong>
- We select the features by manual evaluation (delete malformed or irrelevant features, delete hidden labels), and using a features selection algorithm on the Tshark features to keep only the most useful for the detection.

## Dataset Processing and Evaluation

In [208]:
import pandas as pd
import numpy as np
import os
import glob
from ast import literal_eval
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier

In [209]:
# PATH = '/data/puccetti/space_data/final_merging_all_feature_ordered.csv'
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

input_dir = os.path.join(root_dir, "rospace_dataset", "3_complete_dataset")
output_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")
os.makedirs(output_dir, exist_ok=True)

# 自動抓取 3_complete_dataset 裡最新的 cleaned_merged 檔案
search_pattern = os.path.join(input_dir, "cleaned_merged-*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    raise FileNotFoundError(f"找不到檔案，請確認 {input_dir} 中有資料！")

latest_csv = max(csv_files, key=os.path.getmtime)
PATH = latest_csv
date_val = os.path.basename(latest_csv).replace("cleaned_merged-", "").replace(".csv", "")

print(f"自動載入檔案: {PATH}")
print("-" * 40)

自動載入檔案: c:\Users\yuyux\OneDrive\Desktop\專題\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\3_complete_dataset\cleaned_merged-0512.csv
----------------------------------------


In [210]:
pd.set_option("display.max_columns", None)
pd.get_option("display.max_columns")

In [211]:
df = pd.read_csv(PATH, nrows=5000000, low_memory=False)

In [212]:
df = df.sort_values('timestamp')

In [213]:
# print(df['timestamp'])
print("Timestamp head/tail:\n", df['timestamp'])

Timestamp head/tail:
 0        1.778585e+09
1        1.778585e+09
2        1.778585e+09
3        1.778585e+09
4        1.778585e+09
             ...     
23228    1.778587e+09
23229    1.778587e+09
23230    1.778587e+09
23231    1.778587e+09
23232    1.778587e+09
Name: timestamp, Length: 23233, dtype: float64


In [214]:
# print(df.shape)
print("Shape:", df.shape)

Shape: (23233, 519)


## Some checks on columns and values

In [215]:
# print(df['attack'].value_counts())
print("Attack Value Counts:\n", df['attack'].value_counts())

Attack Value Counts:
 attack
ros2 reconnaissance     11760
observe                 10636
nmap port scanning        501
nmap SYN flood            298
metasploit SYN flood       24
nmap discovery             10
ros2 reflection             2
ros2 node crashing          2
Name: count, dtype: int64


Delete some unuseful columns: 
- 'Unnamed' columns are just duplicate indexes of dataframes

In [216]:
subs = "Unnamed"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [217]:
print(df.columns)

Index(['timestamp', 'layers.frame.frame.time', 'layers.frame.frame.time_utc',
       'layers.frame.frame.time_relative', 'layers.frame.frame.number',
       'layers.frame.frame.len', 'layers.frame.frame.cap_len',
       'layers.frame.frame.protocols', 'layers.ip.ip.dsfield',
       'layers.ip.ip.dsfield_tree.ip.dsfield.dscp',
       ...
       'Tcp_Close', 'nr_active_file', 'nr_inactive_file', 'topic_name',
       'src_topic', 'subscribers_count', 'publisher_count', 'msg_type',
       'msg_data', 'attack'],
      dtype='str', length=519)


## Delete Features related to time 
If we want to train an detector with a shuffled dataset (we want just to distinguish between normal and attack data points, without considering the chronological order of data point occurrences) we have to delete the features related to time as they mark attacks and normal behavoir and are not generalizable (hidden label).

However, we keep the timestamp to use it for time series analysis. The time stamp will be dropped based on the detector that we want to build.  

In [218]:
subs = "time"
res = [i for i in df.columns if subs in i and i != 'timestamp']
print(len(res))
print(res)
df=df.drop(res, axis=1)

9
['layers.frame.frame.time', 'layers.frame.frame.time_utc', 'layers.frame.frame.time_relative', 'layers.udp.Timestamps.udp.time_relative', 'layers.udp.Timestamps.udp.time_delta', 'layers.frame.frame.time_delta', 'layers.frame.frame.time_delta_displayed', 'layers.dns.dns.time', 'layers.icmp.ntp.ntp.reftime']


In [219]:
print(df.shape)

(23233, 510)


### Delete features that specify Source or Destination at diffrent layers of the protocol stack
The model generalization could be degradated by knowledge related to specific values observed during the monitoring campaign. The source and destination addresses are not generalizable, then, we drop them. 

In [220]:
subs = "dst"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

9
['layers.ip.ip.dst', 'layers.ip.ip.dst_host', 'layers.udp.udp.dstport', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.message', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.severity', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.group', 'layers.icmp.ip.ip.dst', 'layers.icmp.ip.ip.dst_host', 'layers.icmp.udp.udp.dstport']


In [221]:
print(df.shape)

(23233, 501)


In [222]:
subs = "src"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

17
['layers.ip.ip.src', 'layers.ip.ip.src_host', 'layers.udp.udp.srcport', 'layers.rtps.rtps.guidPrefix.src', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.hostId', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.appId', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.sm.guidPrefix.instanceId', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.message', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.severity', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.group', 'layers.icmp.ip.ip.src', 'layers.icmp.ip.ip.src_host', 'layers.icmp.udp.udp.srcport', 'layers.icmp.rtps.rtps.guidPrefix.src', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.hostId', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.appId', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.sm.guidPrefix.instanceId']


In [223]:
print(df.shape)

(23233, 484)


In [224]:
subs = "host"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

4
['layers.ip.ip.host', 'layers.rtps.rtps.sm.id_tree.serializedData.serializedData:.PID_PARTICIPANT_GUID.rtps.param.participant_guid_tree.rtps.param.guid.hostId', 'layers.icmp.ip.ip.host', 'layers.icmp.rtps.rtps.sm.id_tree.serializedData.serializedData:.PID_PARTICIPANT_GUID.rtps.param.participant_guid_tree.rtps.param.guid.hostId']


In [225]:
print(df.shape)

(23233, 480)


In [226]:
subs = "addr"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

2
['layers.ip.ip.addr', 'layers.icmp.ip.ip.addr']


In [227]:
print(df.shape)

(23233, 478)


### Delete features related to the "Frame" protocol
From wireshark doc (https://wiki.wireshark.org/Protocols/frame):

"The frame protocol isn't a real protocol itself, but used by Wireshark as a base for all the protocols on top of it. It shows information from capturing, such as the exact time a specific frame was captured. You could think of it as a pseudo dissector."

In [228]:
subs = ".frame."
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

4
['layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.protocols']


In [229]:
print(df.shape)

(23233, 474)


### Delete features that contains ID keyword
We want the dataset to be more general as possible. We drop the ID wich are specific to the execution of the system during the monitoring campaign. Also, the ID can implicitly be an hidden label. For example, the attacker can be associated, during the training to a specific id. However, at test time the association can be different, degrading the performance of the model.

First, the features are printed to ensure that we do not drop features with substring "id" in the name that are relevant.

In [230]:
subs = "id"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

249
['layers.ip.ip.id', 'layers.rtps.rtps.guidPrefix', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.domain_id', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.participant_idx', 'layers.rtps.rtps.sm.id', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.reserved', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.data.serialized_key', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.data_present', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.inline_qos', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.endianness', 'layers.rtps.rtps.sm.id_tree.rtps.sm.octetsToNextHeader', 'layers.rtps.rtps.sm.id_tree.rtps.extra_flags', 'layers.rtps.rtps.sm.id_tree.rtps.octets_to_inline_qos', 'layers.rtps.rt

In [231]:
print(df.shape)

(23233, 225)


In [232]:
subs = "port"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

34
['layers.udp.udp.port', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.rtps.Default port mapping: domainId=Unknown, participantIdx=120, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=1, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.dns.Queries._http._tcp.ports.ubuntu.com: type SRV, class IN.dns.qry.name', 'layers.dns.Queries._http._tcp.ports.ubuntu.com: type SRV, class IN.dns.qry.name.len', 'layers.dns.Queries._http._tcp.ports.ubuntu.com: type SRV, class IN.dns.count.labels', 'layers.dns.Queries._http._tcp.ports.ubuntu.com: type SRV, class IN.dns.qry.type', 'layers.dns.Queries._http._tcp.ports.ubuntu.com: type SRV, class IN.dns.qry.class', 'layers.dns.Queries.ports.ubuntu.com: type A, class IN.dns.qry.name', 'layers.dns.Queries.p

In [233]:
print(df.shape)

(23233, 191)


### Delete malformed features
These features seems to be badly formatted and can be the results of a formatting exeption during the captures. 

In [234]:
subs = "ubuntu"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

20
['layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.name', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.name.len', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.count.labels', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.type', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.class', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.name', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.name.len', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.count.labels', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.type', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.class', 'layers.icmp.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.name', 'layers.icmp.dns.Queries.connectivity-check.u

In [235]:
print(df.shape)

(23233, 171)


In [236]:
subs = "microsoft"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [237]:
print(df.shape)

(23233, 171)


### Manual evaluation of the Tshark features 

In [238]:
subs = "."
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
#df=df.drop(res, axis=1)

138
['layers.ip.ip.dsfield', 'layers.ip.ip.dsfield_tree.ip.dsfield.dscp', 'layers.ip.ip.len', 'layers.ip.ip.flags', 'layers.ip.ip.flags_tree.ip.flags.df', 'layers.ip.ip.ttl', 'layers.ip.ip.proto', 'layers.ip.ip.checksum', 'layers.ip.ip.stream', 'layers.udp.udp.length', 'layers.udp.udp.checksum', 'layers.udp.udp.checksum.status', 'layers.udp.udp.stream', 'layers.udp.udp.stream.pnum', 'layers.udp.udp.payload', 'layers.rtps.rtps.magic', 'layers.rtps.rtps.version', 'layers.rtps.rtps.version_tree.rtps.version.major', 'layers.rtps.rtps.version_tree.rtps.version.minor', 'layers.rtps.rtps.vendorId', 'layers.dns.dns.flags', 'layers.dns.dns.flags_tree.dns.flags.response', 'layers.dns.dns.flags_tree.dns.flags.opcode', 'layers.dns.dns.flags_tree.dns.flags.truncated', 'layers.dns.dns.flags_tree.dns.flags.recdesired', 'layers.dns.dns.flags_tree.dns.flags.z', 'layers.dns.dns.flags_tree.dns.flags.ad', 'layers.dns.dns.flags_tree.dns.flags.checkdisable', 'layers.dns.dns.count.queries', 'layers.dns.dns.c

### Drop the malformed features 
We drop the features with "/" or "\"

In [239]:
subs = "/"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [240]:
subs = "\\"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [241]:
print(df.shape)

(23233, 171)


In [242]:
subs = "PTR"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

12
['layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.name', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.name.len', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.count.labels', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.type', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.class', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.qu', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.name', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.name.len', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.count.labels', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.type', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.class', 'layers.mdns.Queri

In [243]:
print(df.shape)

(23233, 159)


In [244]:
subs = "full_uri"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [245]:
subs = "request_number"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [246]:
subs = "<Root>"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

18
['layers.dns.Additional records.<Root>: type OPT.dns.resp.name', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.type', 'layers.dns.Additional records.<Root>: type OPT.dns.rr.udp_payload_size', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.ext_rcode', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.edns0_version', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z_tree.dns.resp.z.do', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z_tree.dns.resp.z.reserved', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.len', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.name', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.type', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.rr.udp_payload_size', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.ext_rcode', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.edns0_ver

In [247]:
print(df.shape)

(23233, 141)


In [248]:
subs = "len"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

5
['layers.ip.ip.len', 'layers.udp.udp.length', 'layers.icmp.ip.ip.hdr_len', 'layers.icmp.ip.ip.len', 'layers.icmp.udp.udp.length']


In [249]:
print(df.shape)

(23233, 136)


In [250]:
subs = "seq"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


### Save list of features 

In [251]:
features = df.columns

In [252]:
dict = {'features': features}
     
df_features = pd.DataFrame(dict)

In [253]:
# df_features.to_csv("/data/puccetti/space_data/features_usable_temp.csv")
feat_temp_path = os.path.join(output_dir, f"features_usable_temp_{date_val}.csv")
df_features.to_csv(feat_temp_path, index=False)

### Create dataset with the subset of the features 
The objecftive is to understand the memory occupation of the resulting dataset

In [254]:
# PATH = '/data/puccetti/space_data/final_merging_all_feature_ordered.csv'

In [255]:
# features = pd.read_csv("/data/puccetti/space_data/features_usable_temp.csv")
features = pd.read_csv(feat_temp_path)
to_load = features['features'].values.tolist()

In [256]:
df = pd.read_csv(PATH, usecols=to_load, low_memory=False)

In [257]:
# print(df.shape)
print("Shape after reloading:", df.shape)

Shape after reloading: (23233, 136)


In [258]:
# df.to_csv("/data/puccetti/space_data/usable_temp.csv")
usable_temp_path = os.path.join(output_dir, f"usable_temp_{date_val}.csv")
df.to_csv(usable_temp_path, index=False)

In [259]:
# df = pd.read_csv("/data/puccetti/space_data/usable_temp.csv")
df = pd.read_csv(usable_temp_path, low_memory=False)

# Prepare the data for training
In this section, we prepare the data to be processed by ML algorithms. In particular, we perform the following steps:
- <strong>Convert mixed dtypes </strong>: we uniform the type of features with mixed type values.
- <strong>Handle NaN values</strong>: we replace NaN and infinite values with -1. 
- <strong>Convert Label to Numeric</strong>: We substitute label values with numeric values. Then, we create two versions of the dataset: with  binary labels (attack, normal), and with multiple labels (one label for each attack).
- <strong>Convert String To Numeric</strong>: we convert the string values to numbers using categorical encoding. This technique assigns a unique number to any unique string values of a feature.
- <strong>Split the dataset in Training and Test sets</strong>: after removing labels and timestamps columns, we split the dataframe in training and test sets with a 60/40 split.

## Convert mixed dtypes
We convert mixed dtypes columns to string using the following lambda function:

In [260]:
def convert_dtype(x):
    # if not x:
    if pd.isna(x) or x == '':
        return ''
    try:
        return str(x)   
    except:        
        return ''
    
def convert_hex(x):
    # if not x:
    if pd.isna(x) or x == '':
        return 0
    try:
        return literal_eval(x)
    except:        
        return 0

In [261]:
#Indexes of the columns to be converted
# to_convert = [7,10,17,21,31,35,38,41,42,45,49,53,62,64,65,66,69,81,82,85,86,87,90,91,94,96,103,106,111,113,116,120,122,124,127,134,136,139,162,173,176,178,181,184,190,195,197,198,199,225,228,229]
to_convert_cols = df.select_dtypes(include=['object', 'str']).columns

In [262]:
#Convert
# for i in to_convert:
#     df[df.columns[i]] = df[df.columns[i]].apply(lambda x: convert_dtype(x))
for col in to_convert_cols:
    if col != 'attack':
        df[col] = df[col].apply(lambda x: convert_dtype(x))

In [263]:
# print(df.shape)
print("Shape after dtype conversion:", df.shape)

Shape after dtype conversion: (23233, 136)


### Handling NaN values

In [264]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [265]:
print(nanv)

[np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.True_, np.True_, np.True_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.False_, np.True_, np.False_, np.True_, np.False_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.False_, np.True_, np.True_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.False_, np.False_, np.False_, np.True_, np.True_, np.Tru

In [266]:
df.replace([np.inf, -np.inf], -1, inplace=True)
df.fillna(-1, inplace=True)
df=df.dropna(thresh=1, axis=1)

In [267]:
df.replace('nan', -1, inplace=True)

,timestamp,layers.ip.ip.dsfield,layers.ip.ip.dsfield_tree.ip.dsfield.dscp,layers.ip.ip.flags,layers.ip.ip.flags_tree.ip.flags.df,layers.ip.ip.ttl,layers.ip.ip.proto,layers.ip.ip.checksum,layers.ip.ip.stream,layers.udp.udp.checksum,layers.udp.udp.checksum.status,layers.udp.udp.stream,layers.udp.udp.stream.pnum,layers.udp.udp.payload,layers.rtps.rtps.magic,layers.rtps.rtps.version,layers.rtps.rtps.version_tree.rtps.version.major,layers.rtps.rtps.version_tree.rtps.version.minor,layers.rtps.rtps.vendorId,layers.dns.dns.flags,layers.dns.dns.flags_tree.dns.flags.response,layers.dns.dns.flags_tree.dns.flags.opcode,layers.dns.dns.flags_tree.dns.flags.truncated,layers.dns.dns.flags_tree.dns.flags.recdesired,layers.dns.dns.flags_tree.dns.flags.z,layers.dns.dns.flags_tree.dns.flags.ad,layers.dns.dns.flags_tree.dns.flags.checkdisable,layers.dns.dns.count.queries,layers.dns.dns.count.answers,layers.dns.dns.count.auth_rr,layers.dns.dns.count.add_rr,layers.ip.ip.ttl_tree._ws.expert._ws.expert.message,layers.ip.ip.ttl_tree._ws.expert._ws.expert.severity,layers.ip.ip.ttl_tree._ws.expert._ws.expert.group,layers.dns.dns.flags_tree.dns.flags.authoritative,layers.dns.dns.flags_tree.dns.flags.recavail,layers.dns.dns.flags_tree.dns.flags.authenticated,layers.dns.dns.flags_tree.dns.flags.rcode,layers.dns.dns.response_to,layers.mdns.dns.flags,layers.mdns.dns.flags_tree.dns.flags.response,layers.mdns.dns.flags_tree.dns.flags.opcode,layers.mdns.dns.flags_tree.dns.flags.truncated,layers.mdns.dns.flags_tree.dns.flags.recdesired,layers.mdns.dns.flags_tree.dns.flags.z,layers.mdns.dns.flags_tree.dns.flags.checkdisable,layers.mdns.dns.count.queries,layers.mdns.dns.count.answers,layers.mdns.dns.count.auth_rr,layers.mdns.dns.count.add_rr,layers.icmp.icmp.type,layers.icmp.icmp.type_tree._ws.expert._ws.expert.message,layers.icmp.icmp.type_tree._ws.expert._ws.expert.severity,layers.icmp.icmp.type_tree._ws.expert._ws.expert.group,layers.icmp.icmp.code,layers.icmp.icmp.checksum,layers.icmp.icmp.checksum.status,layers.icmp.icmp.unused,layers.icmp.ip.ip.version,layers.icmp.ip.ip.dsfield,layers.icmp.ip.ip.dsfield_tree.ip.dsfield.dscp,layers.icmp.ip.ip.dsfield_tree.ip.dsfield.ecn,layers.icmp.ip.ip.flags,layers.icmp.ip.ip.flags_tree.ip.flags.rb,layers.icmp.ip.ip.flags_tree.ip.flags.df,layers.icmp.ip.ip.flags_tree.ip.flags.mf,layers.icmp.ip.ip.frag_offset,layers.icmp.ip.ip.ttl,layers.icmp.ip.ip.proto,layers.icmp.ip.ip.checksum,layers.icmp.ip.ip.checksum.status,layers.icmp.ip.ip.stream,layers.icmp.udp.udp.checksum,layers.icmp.udp.udp.checksum.status,layers.icmp.udp.udp.stream,layers.icmp.udp.udp.payload,layers.icmp.rtps.rtps.magic,layers.icmp.rtps.rtps.version,layers.icmp.rtps.rtps.version_tree.rtps.version.major,layers.icmp.rtps.rtps.version_tree.rtps.version.minor,layers.icmp.rtps.rtps.vendorId,layers.icmp.ntp.ntp.flags,layers.icmp.ntp.ntp.flags_tree.ntp.flags.li,layers.icmp.ntp.ntp.flags_tree.ntp.flags.vn,layers.icmp.ntp.ntp.flags_tree.ntp.flags.mode,layers.icmp.ntp.ntp.stratum,layers.icmp.ntp.ntp.ppoll,layers.icmp.ntp.ntp.precision,layers.icmp.ntp.ntp.rootdelay,layers.icmp.ntp.ntp.rootdispersion,layers.icmp.ntp.ntp.org,layers.icmp.ntp.ntp.rec,layers.icmp.ntp.ntp.xmt,layers.icmp.dns.dns.flags,layers.icmp.dns.dns.flags_tree.dns.flags.response,layers.icmp.dns.dns.flags_tree.dns.flags.opcode,layers.icmp.dns.dns.flags_tree.dns.flags.truncated,layers.icmp.dns.dns.flags_tree.dns.flags.recdesired,layers.icmp.dns.dns.flags_tree.dns.flags.z,layers.icmp.dns.dns.flags_tree.dns.flags.checkdisable,layers.icmp.dns.dns.count.queries,layers.icmp.dns.dns.count.answers,layers.icmp.dns.dns.count.auth_rr,layers.icmp.dns.dns.count.add_rr,MemFree,Buffers,Cached,Active,Inactive,SwapFree,pgpgin,pgpgout,pgalloc_dma,pgfree,pgactivate,pgdeactivate,pgfault,pgmajfault,Disk_Read,Disk_Write,Net_Sent,Net_Received,Tcp_Listen,Tcp_Established,Tcp_Syn,Tcp_TimeWait,Tcp_Close,nr_active_file,nr_inactive_file,topic_name,src_topic,subscribers_count,publisher_count,msg_type,msg_data,attack
0,1.778585e+09,0x00,0

In [268]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())
print(nanv)

[np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_

In [269]:
#Save the processed dataset
# df.to_csv('/data/puccetti/space_data/usable_temp_nan.csv')
#df = pd.read_csv('/data/puccetti/space_data/final_full_dataset_nan.csv')
usable_temp_nan_path = os.path.join(output_dir, f"usable_temp_nan_{date_val}.csv")
df.to_csv(usable_temp_nan_path, index=False)

### Convert Label columns to Numeric values
We substitute label values with numeric values. We create two version of the dataset:
- binary classification
- multiple label classification

In [270]:
print(df['timestamp'])

0        1.778585e+09
1        1.778585e+09
2        1.778585e+09
3        1.778585e+09
4        1.778585e+09
             ...     
23228    1.778587e+09
23229    1.778587e+09
23230    1.778587e+09
23231    1.778587e+09
23232    1.778587e+09
Name: timestamp, Length: 23233, dtype: float64


In [271]:
# print(df['attack'].value_counts())
print("Attack counts before mapping:\n", df['attack'].value_counts())



Attack counts before mapping:
 attack
ros2 reconnaissance     11760
observe                 10636
nmap port scanning        501
nmap SYN flood            298
metasploit SYN flood       24
nmap discovery             10
ros2 reflection             2
ros2 node crashing          2
Name: count, dtype: int64


In [272]:
df['attack'] = df['attack'].replace('metasploit SYN flood', 1) 
df['attack'] = df['attack'].replace('nmap discovery', 2)
df['attack'] = df['attack'].replace('nmap SYN flood', 3) 
df['attack'] = df['attack'].replace('ros2 node crashing', 4)
df['attack'] = df['attack'].replace('ros2 reconnaissance', 5)
df['attack'] = df['attack'].replace('ros2 reflection', 6)
df['attack'] = df['attack'].replace('nmap port scanning', 7)
df['attack'] = df['attack'].replace('observe', 0)

# df['attack'] = pd.to_numeric(df['attack'])
df['attack'] = pd.to_numeric(df['attack'], errors='coerce').fillna(0).astype(int)

df['attack'].unique(), df['attack'].nunique()

(array([2, 0, 7, 5, 3, 6, 4, 1]), 8)

In [273]:
# print(df['attack'].value_counts())
print("Attack counts after multi-mapping:\n", df['attack'].value_counts())

Attack counts after multi-mapping:
 attack
5    11760
0    10636
7      501
3      298
1       24
2       10
6        2
4        2
Name: count, dtype: int64


In [274]:
# df.to_csv('/data/puccetti/space_data/usable_temp_multi.csv')
usable_temp_multi_path = os.path.join(output_dir, f"usable_temp_multi_{date_val}.csv")
df.to_csv(usable_temp_multi_path, index=False)

In [275]:
df['attack'] = df['attack'].replace(2, 1)
df['attack'] = df['attack'].replace(3, 1) 
df['attack'] = df['attack'].replace(4, 1)
df['attack'] = df['attack'].replace(5, 1)
df['attack'] = df['attack'].replace(6, 1)
df['attack'] = df['attack'].replace(7, 1)

In [276]:
# print(df['attack'].value_counts())
print("Attack counts after binary-mapping:\n", df['attack'].value_counts())

Attack counts after binary-mapping:
 attack
1    12597
0    10636
Name: count, dtype: int64


In [277]:
# df.to_csv('/data/puccetti/space_data/usable_temp_bin.csv')
usable_temp_bin_path = os.path.join(output_dir, f"usable_temp_bin_{date_val}.csv")
df.to_csv(usable_temp_bin_path, index=False)

## Convert String to Numeric

In [278]:
list_column_string=df.select_dtypes(exclude=[np.number]).columns

for i in list_column_string:
    if i != 'timestamp':
        df[i] = pd.Categorical(df[i])

In [279]:
for i in list_column_string:
    if i != 'timestamp':
        df[i] = df[i].cat.codes

### Split the dataset to create Train and Test Sets

In [280]:
from sklearn.model_selection import train_test_split

In [281]:
df_clean = df.copy()

In [282]:
#df = df.drop(['Unnamed: 0'], axis=1)
df = df.drop(['timestamp'], axis=1)
print("Dataset shape before ExtraTrees: " + str(df.shape))

Dataset shape before ExtraTrees: (23233, 135)


In [283]:
subs = "Unnamed"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


I want to make the feature selection only on the feature related to the network monitor (Tshark).

In [284]:
subs = "."
res = [i for i in df.columns if subs in i]
print(len(res))

103


In [285]:
label = df['attack']
df = df.drop(['attack'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(df[res], label, test_size=0.4, random_state=42)

x_train = x_train.to_numpy()
x_test = x_test.to_numpy()

In [286]:
print("Train Set Shape: " + str(x_train.shape))
print("Train Set Label Shape: " + str(y_train.shape))
print("Test Set Shape: " + str(x_test.shape))
print("Test Set Label Shape: " + str(y_test.shape))

Train Set Shape: (13939, 103)
Train Set Label Shape: (13939,)
Test Set Shape: (9294, 103)
Test Set Label Shape: (9294,)


# Select best features for the light version of the dataset
We use the feature ranking algorithm of ExtraTreesClassifier to select the best Tshark features. 

In [287]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel

In [288]:
# clf = ExtraTreesClassifier(n_estimators=30)
clf = ExtraTreesClassifier(n_estimators=30, random_state=42)
clf = clf.fit(x_train, y_train)
clf.feature_importances_

array([3.22430216e-05, 9.85618970e-05, 7.10105111e-05, 0.00000000e+00,
       1.31510719e-03, 2.31682550e-07, 1.96903247e-01, 1.60448567e-03,
       1.29210906e-02, 3.28843802e-05, 4.06090839e-01, 9.14854895e-02,
       5.76918611e-02, 1.06994918e-02, 3.07720419e-03, 6.06376257e-03,
       1.80022477e-02, 6.16128526e-03, 2.37717231e-03, 9.81400267e-04,
       6.92869191e-03, 5.41315004e-03, 2.92494208e-03, 8.40218122e-03,
       1.04156935e-03, 0.00000000e+00, 5.87764106e-03, 8.77043337e-03,
       3.13951454e-05, 2.92347779e-03, 2.22263025e-04, 2.71314370e-03,
       1.44873275e-03, 1.06240416e-03, 9.88053831e-04, 5.65691016e-05,
       1.20214368e-04, 1.27676766e-01, 6.14622272e-06, 1.46428510e-06,
       0.00000000e+00, 2.66637515e-06, 6.93890258e-06, 4.55515810e-06,
       3.16395120e-06, 1.23059267e-05, 7.25208155e-06, 0.00000000e+00,
       7.21215910e-06, 3.20183597e-05, 0.00000000e+00, 1.26615446e-05,
       0.00000000e+00, 0.00000000e+00, 8.52523388e-04, 0.00000000e+00,
      

In [289]:
importances = clf.feature_importances_
indices = np.argsort(importances)[-30:]

In [290]:
print(indices)

[54 19 34 24 33  4 71 32  7 18 31 68 29 22 14 21 26 15 17 20 23 27 13  8
 16 12 11 37  6 10]


In [291]:
best_features = df[res].columns[indices]

In [292]:
subs = "."
# res = [i for i in df.columns if subs not in i]
res_no_dot = [i for i in df.columns if subs not in i]
print(len(res_no_dot))

31


In [293]:
best_features = set(best_features)
# res = set(res)
res_no_dot = set(res_no_dot)
union = list(best_features.union(res_no_dot))

In [294]:
print(union)
print(len(union))

['layers.dns.dns.flags_tree.dns.flags.authoritative', 'layers.dns.dns.response_to', 'Disk_Write', 'Active', 'layers.icmp.udp.udp.checksum', 'Inactive', 'layers.dns.dns.flags_tree.dns.flags.recdesired', 'layers.dns.dns.flags_tree.dns.flags.response', 'Tcp_Established', 'Buffers', 'MemFree', 'pgfree', 'layers.rtps.rtps.version', 'layers.ip.ip.checksum', 'pgalloc_dma', 'Net_Received', 'pgpgin', 'layers.ip.ip.ttl', 'layers.rtps.rtps.magic', 'layers.rtps.rtps.version_tree.rtps.version.minor', 'publisher_count', 'pgactivate', 'Tcp_Close', 'layers.udp.udp.checksum', 'src_topic', 'Tcp_TimeWait', 'msg_data', 'SwapFree', 'nr_inactive_file', 'nr_active_file', 'layers.rtps.rtps.version_tree.rtps.version.major', 'pgmajfault', 'layers.dns.dns.count.answers', 'Cached', 'layers.dns.dns.flags_tree.dns.flags.opcode', 'layers.udp.udp.stream', 'layers.icmp.ip.ip.checksum', 'subscribers_count', 'layers.dns.dns.count.add_rr', 'layers.dns.dns.flags_tree.dns.flags.ad', 'layers.icmp.icmp.checksum', 'Tcp_Syn', 

In [295]:
# np.save('/data/puccetti/space_data/usable_features.npy', union, allow_pickle=True)
usable_features_npy_path = os.path.join(output_dir, f"usable_features_{date_val}.npy")
np.save(usable_features_npy_path, union, allow_pickle=True)

# Compose the reduced dataset

In [296]:
# features = np.load('/data/puccetti/space_data/usable_features.npy')
features = np.load(usable_features_npy_path, allow_pickle=True)

In [297]:
features = list(features)

In [298]:
features.append('attack')

In [299]:
if 'timestamp' in df_clean.columns:
    features.append('timestamp')

In [300]:
# df = pd.read_csv(PATH, usecols=features, low_memory=False)
features_to_keep = [c for c in features if c in df_clean.columns]
df_final = df_clean[features_to_keep]

In [301]:
# print(df['attack'])
print("Final Attack counts:\n", df_final['attack'].value_counts())

Final Attack counts:
 attack
1    12597
0    10636
Name: count, dtype: int64


In [302]:
# df.to_csv('/data/puccetti/space_data/reduced_final.csv')
reduced_final_path = os.path.join(output_dir, f"reduced_final_{date_val}.csv")
df_final.to_csv(reduced_final_path, index=False)
print(f"\n✅ 成功！降維後的最終資料已存至: {reduced_final_path}")


✅ 成功！降維後的最終資料已存至: c:\Users\yuyux\OneDrive\Desktop\專題\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\4_reduced_and_noperiodicity_dataset\reduced_final_0512.csv


In [303]:
#df = pd.read_csv('/data/puccetti/space_data/usable_temp_bin.csv', usecols=features)

In [304]:
#df.to_csv('/data/puccetti/space_data/usable_final_bin.csv')

In [305]:
#df = pd.read_csv('/data/puccetti/space_data/usable_temp_multi.csv', usecols=features)

In [306]:
#df.to_csv('/data/puccetti/space_data/usable_final_multi.csv')

In [307]:
import pandas as pd
import numpy as np
import glob
import os

# ==========================================
# 1. 自動定位路徑與最新檔案
# ==========================================
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

# 定位到第四步的資料夾
check_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")

# 自動搜尋最新的 reduced_final 檔案
search_pattern = os.path.join(check_dir, "reduced_final_*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    print(f"❌ 錯誤：在 {check_dir} 找不到任何 reduced_final 檔案！")
else:
    # 抓取最後修改的檔案
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"🚀 [啟動檢查] 檔案名稱: {os.path.basename(latest_csv)}")
    print("-" * 50)

    # 讀取檔案
    df = pd.read_csv(latest_csv, low_memory=False)

    # --- 檢查 1：檔案維度 ---
    print(f"📊 [1. 維度檢查]")
    print(f"   - 資料筆數 (Rows): {df.shape[0]}")
    print(f"   - 特徵數量 (Columns): {df.shape[1]}")
    if 30 <= df.shape[1] <= 70:
        print("   🟢 狀態: 欄位數量合理（包含 Top 30 網路特徵 + 系統指標）。")
    else:
        print("   ⚠️ 提醒: 欄位數量較多，請確認是否包含過多重複欄位。")

    # --- 檢查 2：幽靈欄位 (Unnamed) ---
    print(f"\n👻 [2. 幽靈欄位檢查]")
    unnamed_cols = [c for c in df.columns if 'Unnamed' in c]
    if len(unnamed_cols) == 0:
        print("   🟢 狀態: 完美！沒有殘留任何 Unnamed 欄位。")
    else:
        print(f"   🔴 錯誤: 發現 {len(unnamed_cols)} 個幽靈欄位: {unnamed_cols}")

    # --- 檢查 3：攻擊標籤 (Label) 純淨度 ---
    print(f"\n🎯 [3. 攻擊標籤檢查]")
    if 'attack' in df.columns:
        counts = df['attack'].value_counts(dropna=False).to_dict()
        print(f"   - 標籤分佈: {counts}")
        
        # 檢查是否只含有 0 和 1 (且必須是數字型態)
        unique_vals = set(df['attack'].unique())
        is_numeric = np.issubdtype(df['attack'].dtype, np.number)
        
        if unique_vals.issubset({0, 1}) and is_numeric:
            print("   🟢 狀態: 完美！標籤已成功轉為數字 0 (正常) 與 1 (攻擊)。")
        else:
            print(f"   🔴 錯誤: 標籤異常！含有非預期數值 {unique_vals} 或型態為 {df['attack'].dtype}")
    else:
        print("   🔴 致命錯誤: 找不到 'attack' 欄位！")

    # --- 檢查 4：遺失值 (NaN) 與 型態 ---
    print(f"\n🧼 [4. 資料品質檢查]")
    nan_count = df.isnull().sum().sum()
    if nan_count == 0:
        print("   🟢 狀態: 完美！全檔案無遺失值 (NaN)。")
    else:
        print(f"   🔴 錯誤: 檔案中還有 {nan_count} 個 NaN！這會導致訓練失敗。")

    # 檢查是否還有字串型態 (除了 timestamp 以外)
    string_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
    # 移除 timestamp，因為它是我們允許的唯一數字字串(或是浮點數)
    if 'timestamp' in string_cols: string_cols.remove('timestamp')
    
    if len(string_cols) == 0:
        print("   🟢 狀態: 完美！所有特徵欄位皆為數值型態。")
    else:
        print(f"   🔴 錯誤: 發現字串欄位: {string_cols} (請在清洗腳本中處理)。")

    print("-" * 50)
    print("✅ 檢查完成！如果上方全是綠燈，你可以放心進入下一步：週期性處理 (No-Periodicity)。")

🚀 [啟動檢查] 檔案名稱: reduced_final_0512.csv
--------------------------------------------------
📊 [1. 維度檢查]
   - 資料筆數 (Rows): 23233
   - 特徵數量 (Columns): 63
   🟢 狀態: 欄位數量合理（包含 Top 30 網路特徵 + 系統指標）。

👻 [2. 幽靈欄位檢查]
   🟢 狀態: 完美！沒有殘留任何 Unnamed 欄位。

🎯 [3. 攻擊標籤檢查]
   - 標籤分佈: {1: 12597, 0: 10636}
   🟢 狀態: 完美！標籤已成功轉為數字 0 (正常) 與 1 (攻擊)。

🧼 [4. 資料品質檢查]
   🟢 狀態: 完美！全檔案無遺失值 (NaN)。
   🟢 狀態: 完美！所有特徵欄位皆為數值型態。
--------------------------------------------------
✅ 檢查完成！如果上方全是綠燈，你可以放心進入下一步：週期性處理 (No-Periodicity)。
